# MedTrack_DV — 01. Data Loading & Profiling

Loads and profiles the raw files for all 3 approved datasets, per the project's Data Finalization Rule
(profile before cleaning; do not skip to cleaning without understanding shape, types, and quality first).

**Datasets in this project:**
1. **HMIS** (19 relational tables) — PRIMARY / BACKBONE dataset. Every row in all 4 final tables originates here.
2. **Beds Management** (4 tables) — SUPPLEMENT dataset. No shared key with HMIS; bridged later via department name + week-to-date mapping.
3. **Readmission dataset** (1 table) — BENCHMARK dataset. No shared key with HMIS; used only for disease-level mortality/readmission/satisfaction benchmarks.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().parent

## 1. HMIS — Hospital HMIS Dataset for Healthcare Analytics

In [2]:
HMIS_RAW_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Hospital HMIS Dataset for Healthcare Analytics"
    / "hospital_synthetic_shalaka"
    / "hospital_data"
)
print(HMIS_RAW_DATA_DIR)

hmis_file_names = [
    # Parent / Master Tables
    "department.csv", "patient.csv", "employee.csv", "disease.csv",
    "insurance_provider.csv", "drug_manufacturer.csv",
    # First-Level Dependent Tables
    "doctor.csv", "ward.csv", "drug.csv", "patient_insurance.csv",
    # Operational Resources
    "bed.csv", "drug_inventory.csv",
    # Hospital Transactions
    "staff_assignment.csv", "admission.csv",
    # Clinical Transactions
    "diagnostic_test.csv", "patient_diagnostic.csv", "prescription.csv",
    # Financial Transactions
    "billing.csv", "billing_detail.csv",
]
for file_name in hmis_file_names:
    file_path = HMIS_RAW_DATA_DIR / file_name
    print(f"\u2713 {file_name} found" if file_path.exists() else f"\u2717 {file_name} NOT found")

C:\Users\sr189\OneDrive\Desktop\Hospital Management Data Analysis\MedTrack_DV\data\raw\Hospital HMIS Dataset for Healthcare Analytics\hospital_synthetic_shalaka\hospital_data
✓ department.csv found
✓ patient.csv found
✓ employee.csv found
✓ disease.csv found
✓ insurance_provider.csv found
✓ drug_manufacturer.csv found
✓ doctor.csv found
✓ ward.csv found
✓ drug.csv found
✓ patient_insurance.csv found
✓ bed.csv found
✓ drug_inventory.csv found
✓ staff_assignment.csv found
✓ admission.csv found
✓ diagnostic_test.csv found
✓ patient_diagnostic.csv found
✓ prescription.csv found
✓ billing.csv found
✓ billing_detail.csv found


In [3]:
def profile_table(df, file_name):
    """Prints a compact but complete profile of one raw table:
    preview, dimensions, one combined column-profile table
    (dtype + non-null/missing counts/% + unique values - avoids
    the redundant overlapping sections an earlier draft had),
    a statistical summary, and a duplicate-row check."""

    display(Markdown("---"))
    display(Markdown(f"# \U0001F4CA Data Profile: `{file_name}`"))

    display(Markdown("## 1. Dataset Preview"))
    display(df.head())

    display(Markdown("## 2. Dataset Dimensions"))
    print(f"Number of Rows    : {df.shape[0]}")
    print(f"Number of Columns : {df.shape[1]}")

    display(Markdown("## 3. Column Profile"))
    column_profile = pd.DataFrame({
        "Data Type": df.dtypes.astype(str),
        "Non-Null Count": df.notnull().sum(),
        "Missing Count": df.isnull().sum(),
        "Missing %": (df.isnull().sum() / len(df) * 100).round(2),
        "Unique Values": df.nunique()
    })
    display(column_profile)

    display(Markdown("## 4. Statistical Summary"))
    display(df.describe(include="all").T)

    display(Markdown("## 5. Duplicate Record Check"))
    duplicate_count = df.duplicated().sum()
    print(f"Duplicate Rows: {duplicate_count}")
    if duplicate_count == 0:
        print("\u2713 No duplicate rows found.")

In [4]:
for file_name in hmis_file_names:
    df = pd.read_csv(HMIS_RAW_DATA_DIR / file_name)
    profile_table(df, file_name)

---

# 📊 Data Profile: `department.csv`

## 1. Dataset Preview

,department_id,department_name,department_type,floor_number,status
0,1,Emergency,Clinical,0,Active
1,2,Internal Medicine,Clinical,2,Active
2,3,Surgery,Clinical,3,Active
3,4,Pediatrics,Clinical,2,Active
4,5,Orthopedics,Clinical,4,Active


## 2. Dataset Dimensions

...

,Data Type,Non-Null Count,Missing Count,Missing %,Unique Values
billing_detail_id,int64,112402,0,0.00,112402
charge_type,object,112402,0,0.00,4
reference_id,float64,45000,67402,59.97,145
amount,int64,112402,0,0.00,31815
bill_id,int64,112402,0,0.00,45000


## 4. Statistical Summary

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
billing_detail_id,112402.0,NaN,NaN,NaN,56201.5,32447.806816,1.0,28101.25,56201.5,84301.75,112402.0
charge_type,112402,4,Room,45000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
reference_id,45000.0,NaN,NaN,NaN,194.5948,113.66573,3.0,98.0,195.0,289.0,414.0
amount,112402.0,NaN,NaN,NaN,13157.063629,16771.971534,200.0,2020.0,4879.0,19000.75,120000.0
bill_id,112402.0,NaN,NaN,NaN,22503.190388,12991.639176,1.0,11270.0,22479.5,33752.0,45000.0


## 5. Duplicate Record Check

Duplicate Rows: 0
✓ No duplicate rows found.


## 2. Beds Management — Hospital Beds Management

In [5]:
BEDS_RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "Hospital Beds Management"
print(BEDS_RAW_DATA_DIR)

beds_file_names = ["patients.csv", "services_weekly.csv", "staff.csv", "staff_schedule.csv"]
for file_name in beds_file_names:
    file_path = BEDS_RAW_DATA_DIR / file_name
    print(f"\u2713 {file_name} found" if file_path.exists() else f"\u2717 {file_name} NOT found")

C:\Users\sr189\OneDrive\Desktop\Hospital Management Data Analysis\MedTrack_DV\data\raw\Hospital Beds Management
✓ patients.csv found
✓ services_weekly.csv found
✓ staff.csv found
✓ staff_schedule.csv found


In [6]:
for file_name in beds_file_names:
    df = pd.read_csv(BEDS_RAW_DATA_DIR / file_name)
    profile_table(df, file_name)

---

# 📊 Data Profile: `patients.csv`

## 1. Dataset Preview

,patient_id,name,age,arrival_date,departure_date,service,satisfaction
0,PAT-09484753,Richard Rodriguez,24,2025-03-16,2025-03-22,surgery,61
1,PAT-f0644084,Shannon Walker,6,2025-12-13,2025-12-14,surgery,83
2,PAT-ac6162e4,Julia Torres,24,2025-06-29,2025-07-05,general_medicine,83
3,PAT-3dda2bb5,Crystal Johnson,32,2025-10-12,2025-10-23,emergency,81
4,PAT-08591375,Garrett Lin,25,2025-02-18,2025-02-25,ICU,76


## 2. Dataset Dimensions

...

,Data Type,Non-Null Count,Missing Count,Missing %,Unique Values
week,int64,6552,0,0.0,52
staff_id,object,6552,0,0.0,126
staff_name,object,6552,0,0.0,126
role,object,6552,0,0.0,3
service,object,6552,0,0.0,4
present,int64,6552,0,0.0,2


## 4. Statistical Summary

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
week,6552.0,NaN,NaN,NaN,26.5,15.009476,1.0,13.75,26.5,39.25,52.0
staff_id,6552,126,STF-b77cdc60,52,NaN,NaN,NaN,NaN,NaN,NaN,NaN
staff_name,6552,126,Allison Hill,52,NaN,NaN,NaN,NaN,NaN,NaN,NaN
role,6552,3,nurse,3796,NaN,NaN,NaN,NaN,NaN,NaN,NaN
service,6552,4,emergency,2028,NaN,NaN,NaN,NaN,NaN,NaN,NaN
present,6552.0,NaN,NaN,NaN,0.599817,0.489973,0.0,0.0,1.0,1.0,1.0


## 5. Duplicate Record Check

Duplicate Rows: 0
✓ No duplicate rows found.


## 3. Readmission — Hospital Data for Patient Readmission Prediction

In [7]:
READMISSION_RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "Hospital Data for Patient Readmission Prediction"
print(READMISSION_RAW_DATA_DIR)   # fixed: previously printed Beds Management's path by mistake

readmission_file_names = ["Healthcare Data Analysis for readmission.csv"]
for file_name in readmission_file_names:
    file_path = READMISSION_RAW_DATA_DIR / file_name
    print(f"\u2713 {file_name} found" if file_path.exists() else f"\u2717 {file_name} NOT found")

C:\Users\sr189\OneDrive\Desktop\Hospital Management Data Analysis\MedTrack_DV\data\raw\Hospital Data for Patient Readmission Prediction
✓ Healthcare Data Analysis for readmission.csv found


In [8]:
for file_name in readmission_file_names:
    df = pd.read_csv(READMISSION_RAW_DATA_DIR / file_name)
    profile_table(df, file_name)

---

# 📊 Data Profile: `Healthcare Data Analysis for readmission.csv`

## 1. Dataset Preview

,hospital_name,Admission_date,hospital_id,hospital_beds_available,occupied_beds,hospital_ward,patient_id,patient_gender,patient_age,patient_race,...,doctor_id,doctor_name,doctor_specialty,patient_assigned_doctor,patient_checkin_date,patient_checkout_date,patient_disease,patient_length_of_stay,discharge_status,readmission
0,The Johns Hopkins Hospital,30-01-2024,3946,260,90,Maternity,1421,Female,48,Asian,...,1725,Justin Morris,Rheumatology,False,14-07-2024,19-07-2024,Anxiety Disorders,6,Deceased,1
1,The Johns Hopkins Hospital,20-03-2022,7147,250,90,ICU,4922,Male,40,Black,...,7510,Latoya Moss,Pulmonology or Allergy and Immunology,False,19-07-2024,07/07/2024,Urinary Tract Infection (UTI),2,Deceased,1
2,The Johns Hopkins Hospital,13-07-2021,4174,350,220,Pediatrics,9804,Female,74,Black,...,7137,Michael Allen,Neurology,False,20-07-2024,29-06-2024,Hypertension (High Blood Pressure),17,Recovered,0
3,The Johns Hopkins Hospital,26-01-2021,2466,190,350,Pediatrics,5622,Male,82,Hispanic,...,8801,Rebecca Mccullough,Cardiology,False,25-06-2024,16-07-2024,Arthritis,28,Recovered,0
4,The Johns Hopkins Hospital,06/02/2024,3014,220,390,Surgery,3314,Male,62,Hispanic,...,3588,John Giles,Rheumatology,False,21-07-2024,19-07-2024,Cancer,16,Transferred,1


## 2. Dataset Dimensions

...

,Data Type,Non-Null Count,Missing Count,Missing %,Unique Values
hospital_name,object,10000,0,0.0,1
Admission_date,object,10000,0,0.0,1460
hospital_id,int64,10000,0,0.0,6074
hospital_beds_available,int64,10000,0,0.0,41
occupied_beds,int64,10000,0,0.0,41
hospital_ward,object,10000,0,0.0,5
patient_id,int64,10000,0,0.0,6022
patient_gender,object,10000,0,0.0,2
patient_age,int64,10000,0,0.0,96
patient_race,object,10000,0,0.0,5


## 4. Statistical Summary

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
hospital_name,10000,1,The Johns Hopkins Hospital,10000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Admission_date,10000,1460,17-09-2020,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hospital_id,10000.0,NaN,NaN,NaN,5515.9666,2605.495574,1000.0,3255.75,5516.5,7790.25,9998.0
hospital_beds_available,10000.0,NaN,NaN,NaN,299.111,117.973859,100.0,200.0,300.0,400.0,500.0
occupied_beds,10000.0,NaN,NaN,NaN,199.085,118.299418,0.0,100.0,200.0,300.0,400.0
hospital_ward,10000,5,Pediatrics,2067,NaN,NaN,NaN,NaN,NaN,NaN,NaN
patient_id,10000.0,NaN,NaN,NaN,5498.7387,2615.168591,1000.0,3232.5,5478.5,7779.5,9999.0
patient_gender,10000,2,Male,5012,NaN,NaN,NaN,NaN,NaN,NaN,NaN
patient_age,10000.0,NaN,NaN,NaN,47.283,27.719692,0.0,23.0,47.0,71.0,95.0
patient_race,10000,5,Hispanic,2042,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Duplicate Record Check

Duplicate Rows: 0
✓ No duplicate rows found.
